# CodeArch real Gemma context test
Private offline 31B experiment. Approved maximum session: 2 hours. No competition submission.
A original; B hide thought history; C B + 8192 output; D C + concise handoffs.
Published tasks only. Model usage, errors, traces, patches and tests are saved.


In [ ]:
import os, time, json, threading, sys, traceback, hashlib, base64, io, zipfile, re
from pathlib import Path
START = time.monotonic()
REMAINING = min(7200, int(1795248120 - time.time()))  # 2026-11-21 08:02 UTC, across retries
assert REMAINING > 300, 'Approved session deadline reached'
DEADLINE = START + REMAINING - 100
WORK = Path('/kaggle/working/codearch-real-gemma')
WORK.mkdir(parents=True, exist_ok=True)
server_instance = None
def save_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
def emergency_stop():
    save_json(WORK/'time-limit.json', {'status':'stopped_at_session_time_limit'})
    os._exit(124)
watchdog = threading.Timer(REMAINING - 50, emergency_stop)
watchdog.daemon = True
watchdog.start()
import torch
gpu = [{'name':torch.cuda.get_device_name(i), 'bytes':torch.cuda.get_device_properties(i).total_memory}
       for i in range(torch.cuda.device_count())]
save_json(WORK/'hardware.json', {'gpus':gpu, 'real_model_inference':False})
print('Hardware:', gpu, flush=True)
assert len(gpu) >= 4 and all('L4' in g['name'] for g in gpu), 'Approved L4 x4 hardware unavailable; stopping without switching models.'


In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


In [ ]:
PAYLOADS = {'A': '...', 'D': '...', 'C': '...'}
EXPECTED_SHA = {'A': 'fa797522dfd3fadcac6d4e46f00a8caf1bdf4cbd4eee8b3585a03db47a3f1bc5', 'D': '0514525a1c2cbb54d668d35bb37c0d1a32cbac1ad220ddf1c8494f2c852ac74a', 'B': '3276d545b70984e508b9e3e0c1ea8ef2e6f2cc6068f2acab4684fc98aebfcc52', 'C': 'cde3d65a6a6acb27a8e003d15bc4595d25bdfdb8aa416f79ace3c00d32f2fcab'}

import yaml
MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other') / MODEL_NAME / '2'
assert MODEL_PATH.is_dir(), 'Approved model weights unavailable; stopping without switching models.'
TASKS_PATH = DATA_DIR/'tasks.jsonl'
GRAPH_DIR, EMBEDDINGS_DIR = str(DATA_DIR/'graphs'), str(DATA_DIR/'embeddings')
from swegemma.models import load_tasks
task_map = {t.instance_id:t for t in load_tasks(TASKS_PATH)}
TASK_IDS = ['rich_3063', 'fastapi_11194', 'requests_6629']
assert all(t in task_map for t in TASK_IDS), 'Published test task missing'
VARIANTS = {}
for name, encoded in PAYLOADS.items():
    content = base64.b64decode(encoded)
    assert hashlib.sha256(content).hexdigest() == EXPECTED_SHA[name]
    folder = WORK/'variants'/name
    folder.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        assert z.testzip() is None
        for info in z.infolist():
            assert not Path(info.filename).is_absolute() and '..' not in Path(info.filename).parts
        z.extractall(folder)
    VARIANTS[name] = folder
save_json(WORK/'variant-manifest.json', EXPECTED_SHA)
print('All four variants and published tasks ready', flush=True)


In [ ]:
from adk_submission import compile_submission, ModelRegistry
from swegemma.config import build_submission_limits
limits, gen_constraints = build_submission_limits()
def run_command(command: str) -> str:
    return 'compile-only'

def read_file(filepath: str, start_line: int | None=None, end_line: int | None=None) -> str:
    return 'compile-only'

def edit_file(filepath: str, old_string: str, new_string: str, allow_multiple: bool=False) -> str:
    return 'compile-only'

def write_file(filepath: str, content: str) -> str:
    return 'compile-only'

def get_code_neighbors(node: str, edge_type: str | None=None, max_neighbors: int=50) -> str:
    return 'compile-only'

def search_similar_code(query: str, k: int=10) -> str:
    return 'compile-only'

def get_code_subgraph(nodes: list[str]) -> str:
    return 'compile-only'

def get_status() -> str:
    return 'compile-only'

def submit_patch() -> str:
    return 'compile-only'
compile_models = ModelRegistry()
compile_models.register(MODEL_NAME, MODEL_NAME)
compile_tools = {f.__name__:f for f in [run_command, read_file, edit_file, write_file,
    get_code_neighbors, search_similar_code, get_code_subgraph, get_status, submit_patch]}
compiled = []
for name, folder in VARIANTS.items():
    agent = compile_submission(folder, tool_registry=compile_tools, model_registry=compile_models,
                               limits=limits, generation_constraints=gen_constraints)
    compiled.append({'variant':name, 'agent':agent.name, 'stage':'compile_only'})
save_json(WORK/'full-compilation.json', compiled)
print('Official ADK full compilation passed for A/B/C/D (no inference yet)', flush=True)


In [ ]:
AGENT_DIR = VARIANTS['A']
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)
save_json(WORK/'model-started.json', {'model':MODEL_NAME,'gpu_count':torch.cuda.device_count(),'base_url':server_instance.base_url})


In [ ]:
import asyncio, concurrent.futures, itertools
from google.adk.models.lite_llm import LiteLlm
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator
CURRENT = {'run':None}
counter = itertools.count(1)
original_generate = LiteLlm.generate_content_async
def dump_content(value):
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json', exclude_none=True)
    return value
def append_event(record):
    with (WORK/'model-calls.jsonl').open('a',encoding='utf-8') as f:
        f.write(json.dumps(record,ensure_ascii=False,default=str)+'\n')
async def observed_generate(self, llm_request, stream=False):
    call_id = next(counter)
    run = CURRENT['run'] or 'setup'
    system = getattr(getattr(llm_request,'config',None),'system_instruction',None)
    system_text = system if isinstance(system,str) else str(system)
    role = next((line for line in system_text.splitlines() if line.startswith('# Role:')), 'unknown')
    contents = [dump_content(c) for c in llm_request.contents]
    save_json(WORK/f'request-{call_id:05d}.json', {'run':run,'role':role,'system':dump_content(system),'contents':contents})
    append_event({'event':'request','run':run,'role':role,'call_id':call_id,
                  'input_characters':len(json.dumps(contents,ensure_ascii=False,default=str)),
                  'time':time.time()})
    try:
        async for response in original_generate(self,llm_request,stream=stream):
            usage = getattr(response,'usage_metadata',None)
            append_event({'event':'response','run':run,'role':role,'call_id':call_id,
                          'usage':dump_content(usage),'time':time.time()})
            yield response
    except Exception as error:
        append_event({'event':'error','run':run,'role':role,'call_id':call_id,
                      'error_type':type(error).__name__,'error':str(error),'time':time.time()})
        raise
LiteLlm.generate_content_async = observed_generate

def run_sync(factory):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda:asyncio.run(factory())).result()

results=[]
def execute(variant, task_id):
    run = variant+'__'+task_id
    folder = WORK/'results'/run
    folder.mkdir(parents=True,exist_ok=True)
    CURRENT['run'] = run
    cfg = EvalConfig(tasks_path=TASKS_PATH,snapshots_dir=DATA_DIR/'snapshots',results_dir=folder,
        submission_dir=VARIANTS[variant],models=models,sandbox='subprocess',
        timeout_seconds=300,verification_timeout_seconds=300,max_time_minutes=5,
        max_tool_calls=60,max_turns=100,limits=limits,generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048,ttl_seconds=1800,cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15,overlap_size=2,
                                                       token_threshold=14336,event_retention_size=5),
        graph_dir=GRAPH_DIR,embeddings_dir=EMBEDDINGS_DIR,wheels_dir=DATA_DIR/'wheels',verbose=False)
    evaluator=Evaluator(cfg)
    print('START',run,flush=True)
    t0=time.monotonic()
    async def evaluate():
        # Includes sandbox setup and verification. Agent's own budget stays 5 min.
        return await asyncio.wait_for(evaluator.evaluate_task(task=task_map[task_id],task_index=1,total_tasks=1),
                                      timeout=min(1000,max(1,DEADLINE-time.monotonic()-30)))
    try:
        result=run_sync(evaluate)
        row={'variant':variant,'task':task_id,'resolved':bool(result.resolved),
             'error':getattr(result,'error',None),'test_exit_code':result.test_exit_code,
             'patch_characters':len(result.agent_patch or ''),'tool_calls':result.tool_calls,
             'llm_calls':getattr(result,'total_llm_calls',None),'wall_seconds':time.monotonic()-t0}
        (folder/'agent.patch').write_text(result.agent_patch or '',encoding='utf-8')
        (folder/'test-output.txt').write_text(result.test_output or '',encoding='utf-8')
    except Exception as error:
        row={'variant':variant,'task':task_id,'error':str(error),'error_type':type(error).__name__,
             'resolved':False,'wall_seconds':time.monotonic()-t0}
        (folder/'exception.txt').write_text(traceback.format_exc(),encoding='utf-8')
    results.append(row)
    save_json(WORK/'results.json',results)
    print('RESULT',json.dumps(row,default=str),flush=True)
    return row

try:
    smoke_a=execute('A',TASK_IDS[0])
    smoke_d=execute('D',TASK_IDS[0])
    # Do not spend more GPU quota if the actual model/tool/patch chain did not run.
    smoke_ok=(smoke_a.get('llm_calls',0) or 0)>0 and (smoke_d.get('llm_calls',0) or 0)>0
    smoke_ok=smoke_ok and smoke_d.get('patch_characters',0)>0 and smoke_d.get('test_exit_code',-1)>=0
    if smoke_ok:
        schedule=[('B',TASK_IDS[0]),('C',TASK_IDS[0])]+[(v,t) for t in TASK_IDS[1:] for v in ['A','B','C','D']]
        for variant,task_id in schedule:
            if time.monotonic()+1050>DEADLINE:
                save_json(WORK/'stopped-early.json',{'reason':'session time budget','remaining_runs':12-len(results)})
                break
            execute(variant,task_id)
    else:
        save_json(WORK/'stopped-early.json',{'reason':'real-model smoke did not complete model/tool/patch/test chain'})
finally:
    LiteLlm.generate_content_async=original_generate
    if server_instance is not None:
        server_instance.stop()
    watchdog.cancel()
    save_json(WORK/'run-finished.json',{'real_model_requested':True,'runs_completed':len(results),
        'elapsed_seconds':time.monotonic()-START,'note':'Inspect response usage to confirm actual model inference; empty fields are not zero tokens.'})
    print('FINISHED',len(results),'runs; inference server stopped',flush=True)
